# aphex on Kaggle

End-to-end smoke test for [aphex](https://github.com/rayansh365/pareto) on a Kaggle notebook.
Targets a GPU runtime (T4 x2 or P100) so the TensorRT / CUDA backends light up; falls back to CPU-only candidates otherwise.

Run order:
1. Install aphex from source.
2. Build a tiny CNN and save it to disk.
3. Generate calibration + eval data and an `--infer-fn` Python file.
4. Walk through the CLI: `analyze` → `preflight` → `benchmark` → `optimize` → `convert` → `check`.
5. Repeat the flow with a **multi-input** transformer-style model (exercises the Phase 2 `input_ids+attention_mask` path).

Everything writes to `/kaggle/working/` so artifacts persist in the notebook output.

## 1. Install aphex

Two options — pick whichever fits your Kaggle setup:

- **Option A (default below):** clone the public repo and `pip install -e`. Needs network access (enable *Internet on* in the notebook sidebar).
- **Option B:** upload the repo as a Kaggle Dataset and point `APHEX_SRC` at `/kaggle/input/<dataset-name>`.

In [ ]:
import os, subprocess, sys, pathlib

APHEX_SRC = pathlib.Path('/kaggle/working/aphex')
REPO_URL  = 'https://github.com/rayansh365/pareto.git'  # update if the repo lives elsewhere

if not APHEX_SRC.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(APHEX_SRC)])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(APHEX_SRC)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'onnx', 'onnxruntime'])

# Confirm the CLI is on PATH and report the version.
subprocess.check_call(['aphex', '--help'], stdout=subprocess.DEVNULL)
print('aphex installed at', APHEX_SRC)

## 2. Build a tiny CNN and save it

We use a 3-layer conv net on 3×32×32 inputs so candidate generation has something interesting to do (FP16, INT8 dynamic, ONNX, etc.) without blowing the Kaggle session timeout.

In [ ]:
import torch, torch.nn as nn, pathlib

WORK = pathlib.Path('/kaggle/working')
WORK.mkdir(exist_ok=True)

class TinyCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Linear(32, num_classes)

    def forward(self, x):
        return self.head(self.features(x))

model = TinyCNN().eval()
MODEL_PT = WORK / 'tiny_cnn.pt'
torch.save(model, MODEL_PT)
print('saved', MODEL_PT, MODEL_PT.stat().st_size, 'bytes')

## 3. Calibration + eval data + inference function

`aphex optimize` needs:
- a labelled eval set (`--eval`) so it can measure quality loss per candidate,
- an inference callable (`--infer-fn`) that turns a list of samples into predictions.

Calibration data is optional but unlocks the INT8 candidates.

In [ ]:
import torch, pathlib
WORK = pathlib.Path('/kaggle/working')

torch.manual_seed(0)
calib = torch.randn(64, 3, 32, 32)
torch.save(calib, WORK / 'calib.pt')

eval_inputs = torch.randn(128, 3, 32, 32)
eval_labels = torch.randint(0, 10, (128,))
torch.save({'inputs': eval_inputs, 'labels': eval_labels}, WORK / 'eval.pt')

INFER_PY = WORK / 'infer.py'
INFER_PY.write_text('''\
import torch
from pathlib import Path

_MODEL = None

def _load():
    global _MODEL
    if _MODEL is None:
        _MODEL = torch.load(Path("/kaggle/working/tiny_cnn.pt"), weights_only=False).eval()
    return _MODEL

def predict(inputs):
    model = _load()
    batch = torch.stack([x if isinstance(x, torch.Tensor) else torch.as_tensor(x) for x in inputs])
    with torch.no_grad():
        logits = model(batch.float())
    return logits.argmax(-1).cpu().numpy()
''')
print('wrote', INFER_PY)

## 4. Walk the CLI

Each cell shells out to the installed `aphex` entry point so the output matches what an engineer sees in their terminal.

In [ ]:
!aphex analyze /kaggle/working/tiny_cnn.pt --input-shape 3,32,32

In [ ]:
!aphex preflight /kaggle/working/tiny_cnn.pt --input-shape 3,32,32

In [ ]:
!aphex benchmark /kaggle/working/tiny_cnn.pt \
  --input-shape 3,32,32 \
  --batch-sizes 1,4 \
  --calibration-data /kaggle/working/calib.pt

### `optimize` — the main event

Picks a Pareto-optimal candidate that respects the latency / quality constraints we pass. The `deployment.yaml` it writes is what `convert` and `check` consume.

In [ ]:
!aphex optimize /kaggle/working/tiny_cnn.pt \
  --input-shape 3,32,32 \
  --batch-sizes 1,4 \
  --calibration-data /kaggle/working/calib.pt \
  --eval /kaggle/working/eval.pt \
  --infer-fn /kaggle/working/infer.py:predict \
  --max-accuracy-loss 0.20 \
  --output /kaggle/working/deployment.yaml \
  --metrics /kaggle/working/metrics.json

In [ ]:
print(open('/kaggle/working/deployment.yaml').read())

### `convert` — write a deployable artifact

Uses `--from-config` so the backend + input shape come straight from `deployment.yaml`. The Phase 1 parity check reloads the artifact and compares outputs to the source model; it exits non-zero on a real divergence.

In [ ]:
!aphex convert /kaggle/working/tiny_cnn.pt \
  --from-config /kaggle/working/deployment.yaml \
  --output /kaggle/working/tiny_cnn.optimized

### `check` — re-validate the chosen config on this hardware

Useful when the optimize run came from a different machine (e.g. a cloud VM via `--remote`) and you want to re-confirm the numbers locally.

In [ ]:
!aphex check /kaggle/working/tiny_cnn.pt \
  --from-config /kaggle/working/deployment.yaml

## 5. Multi-input model — transformer-style

Phase 2 added support for models that take *more than one* tensor (HuggingFace transformers being the canonical case: `input_ids` + `attention_mask`). We exercise it here with a hand-rolled tiny encoder so we don't have to pull a real HF model.

The `--input-shape` syntax: `name:dims:dtype;name:dims:dtype`.

In [ ]:
import torch, torch.nn as nn, pathlib
WORK = pathlib.Path('/kaggle/working')

class TinyEncoder(nn.Module):
    def __init__(self, vocab: int = 1000, dim: int = 32, classes: int = 4):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.norm  = nn.LayerNorm(dim)
        self.head  = nn.Linear(dim, classes)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        x = self.embed(input_ids)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (x * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.head(self.norm(pooled))

enc = TinyEncoder().eval()
MODEL_MULTI = WORK / 'tiny_encoder.pt'
torch.save(enc, MODEL_MULTI)
print('saved', MODEL_MULTI)

In [ ]:
!aphex benchmark /kaggle/working/tiny_encoder.pt \
  --input-shape 'input_ids:64:long;attention_mask:64:long' \
  --batch-sizes 1,4

### Convert the multi-input model

ONNX and PyTorch backends handle multi-input end-to-end. TensorRT / OpenVINO FP32/FP16 conversion is also supported (Phase 2); INT8 multi-input calibration remains gated.

In [ ]:
!aphex convert /kaggle/working/tiny_encoder.pt \
  --backend onnx_cpu \
  --input-shape 'input_ids:64:long;attention_mask:64:long' \
  --output /kaggle/working/tiny_encoder.onnx

In [ ]:
# If the runtime has a CUDA GPU, try the TensorRT FP16 path. This is the
# Phase 2 multi-input TRT route — it builds an engine with one optimization
# profile per input. Skipped automatically when tensorrt is not installed.
import importlib.util, subprocess
if importlib.util.find_spec('tensorrt') is not None:
    subprocess.run([
        'aphex', 'convert', '/kaggle/working/tiny_encoder.pt',
        '--backend', 'tensorrt_fp16',
        '--input-shape', 'input_ids:64:long;attention_mask:64:long',
        '--output', '/kaggle/working/tiny_encoder.engine',
        '--no-verify',  # parity-check requires the TRT runner, not in scope here
    ])
else:
    print('tensorrt not installed in this runtime — skipping TRT multi-input conversion')

## Done

Artifacts in `/kaggle/working/`:
- `deployment.yaml`, `metrics.json` — optimize output
- `tiny_cnn.optimized*` — converted single-input artifact
- `tiny_encoder.onnx` (+ `tiny_encoder.engine` on GPU runtimes) — multi-input artifacts

Next steps you might try on Kaggle hardware that you can't on a laptop:
- `--remote user@host` to drive a separate cloud VM,
- bigger models (e.g. `torchvision.models.resnet50`) to see the Pareto curve open up,
- TensorRT INT8 on single-input models with `--calibration-data`.